这一章主要对应 Go 面试里的五条基础主线：

第一，Go 的错误处理机制。
Java 里有 Exception、RuntimeException、Error、try-catch-finally、throw/throws；Go 不走这条路线，而是显式返回 error。面试时不能只说“Go 没有异常”，而要讲清楚 Go 为什么选择 error value、error wrapping、errors.Is、errors.As，以及项目里如何设计错误码和错误边界。

第二，panic 和 recover。
Go 虽然没有 Java 那种异常体系，但有 panic/recover。它不是常规业务错误处理工具，而是用于不可恢复错误、程序员错误、框架兜底保护。尤其要讲清楚 recover 必须在 defer 中生效，以及 goroutine 之间不能跨协程 recover。

第三，reflect 反射机制。
Java 反射是框架底层基础，Go 反射同样是 JSON、ORM、配置解析、依赖注入、参数校验等框架的底层能力。Go 反射的核心是 Type 和 Value，还要理解 Kind、CanSet、Elem、结构体 tag、interface 底层模型。

第四，Go 泛型。
Go 1.18 引入泛型之后，很多 Java 同学容易拿 Java 泛型的“类型擦除”去类比 Go，但 Go 泛型的设计和实现思路不完全一样。面试重点是类型参数、类型约束、any、comparable、~T、泛型和 interface/reflect 的取舍。

第五，Go IO。
Java 里常考 BIO/NIO/AIO；Go 里则重点考 io.Reader/io.Writer 抽象、bufio、bytes.Buffer、os.File、net.Conn、context 超时取消，以及 Go 的网络 IO 为什么看起来是阻塞写法却能支撑高并发。

这一章最终要形成一句话：

Go 基础不是背语法，而是理解 Go 如何通过“显式错误、defer 资源释放、接口抽象、反射兜底、泛型增强类型安全、Reader/Writer 统一 IO 模型”来构建工程代码。

---

# 一、Go 的 error 机制

## 【文本块 2】Q：Go 是怎么处理错误的？和 Java 异常有什么区别？

Go 的错误处理核心不是 try-catch，而是把错误当成普通返回值显式返回。

最常见形式是：

```go
result, err := doSomething()
if err != nil {
    return err
}
```

Go 标准库里 error 是一个接口：

```go
type error interface {
    Error() string
}
```

也就是说，只要一个类型实现了 `Error() string` 方法，它就可以作为 error 返回。

和 Java 异常相比，Go 的 error 有几个明显特点：

第一，错误是显式的。
函数签名里直接能看到这个函数可能返回 error，调用方必须在代码里处理它。虽然编译器不会像 Java Checked Exception 那样强制捕获，但 Go 代码风格强烈要求显式判断 err。

第二，错误是值。
error 只是一个普通接口值，可以传递、包装、比较、记录、返回。这种设计更简单，也更符合 Go “少魔法”的语言风格。

第三，业务错误和系统错误通常都通过 error 表达。
比如参数错误、数据库错误、网络超时、权限错误，都可以返回 error。真正不可恢复的情况才考虑 panic。

第四，Go 没有异常栈自动传播的语法。
在 Go 中，错误传播通常是手动的：

```go
if err != nil {
    return fmt.Errorf("query user: %w", err)
}
```

这种方式看起来啰嗦，但好处是每一层都能明确补充上下文。

面试里可以这样回答：

Go 的错误处理是显式返回 error，而不是通过异常机制隐式跳转控制流。error 本质是接口，任何实现 Error() string 的类型都可以作为错误。相比 Java 的 try-catch，Go 更强调调用方显式判断、逐层包装上下文，让错误路径在代码中清晰可见。

---

## 【代码块 1】最基本的 error 返回

```go
import (
    "errors"
    "fmt"
)

func divide(a, b int) (int, error) {
    if b == 0 {
        return 0, errors.New("division by zero")
    }
    return a / b, nil
}

result, err := divide(10, 0)
if err != nil {
    fmt.Println("error:", err)
} else {
    fmt.Println("result:", result)
}
```

---

## 【文本块 3】代码解释

这里 `divide` 返回两个值：结果和 error。

如果 b 等于 0，就返回一个非 nil 的 error。
调用方通过 `if err != nil` 判断是否出错。

这就是 Go 里最典型的错误处理方式。

它不像 Java 那样把异常抛出去，而是让错误成为函数返回值的一部分。

---

## 【文本块 4】Q：error 是接口，那 nil error 有什么坑？

这是 Go 面试里非常高频的问题，尤其和 interface 的 nil 陷阱有关。

error 是接口。接口底层可以理解成两部分：

1. 动态类型
2. 动态值

只有当动态类型和动态值都为 nil 时，接口才等于 nil。

如果你把一个具体错误类型的 nil 指针返回成 error，那么这个 error 接口里已经有了动态类型，虽然动态值是 nil，但接口本身不等于 nil。

这会导致调用方判断：

```go
if err != nil
```

结果为 true。

---

## 【代码块 2】error nil 陷阱

```go
import "fmt"

type MyError struct{}

func (e *MyError) Error() string {
    return "my error"
}

func doSomething() error {
    var err *MyError = nil
    return err
}

err := doSomething()

fmt.Println(err == nil)
fmt.Printf("type=%T, value=%#v\n", err, err)
```

---

## 【文本块 5】代码解释

输出中，`err == nil` 是 false。

因为返回值不是一个真正的 nil interface，而是：

```text
type = *MyError
value = nil
```

所以工程里不要返回带类型的 nil。

错误写法：

```go
var err *MyError = nil
return err
```

正确写法：

```go
if noError {
    return nil
}
return &MyError{}
```

面试里可以这样总结：

error 是接口，接口只有动态类型和动态值都为 nil 时才等于 nil。返回自定义错误指针时，要避免把 nil 指针装进 error 接口，否则调用方会误判为非 nil 错误。

---

# 二、创建 error 的几种方式

## 【文本块 6】Q：Go 里创建 error 有哪些方式？

常见方式有三种。

第一，使用 errors.New。

适合创建简单的静态错误：

```go
errors.New("invalid argument")
```

第二，使用 fmt.Errorf。

适合拼接动态信息：

```go
fmt.Errorf("user %d not found", userID)
```

第三，自定义错误类型。

适合携带错误码、业务字段、底层错误等结构化信息。

比如：

```go
type BizError struct {
    Code int
    Msg  string
}
```

如果只是简单错误，用 errors.New 即可。
如果要带上下文，用 fmt.Errorf。
如果项目需要统一错误码、HTTP 状态码、用户提示、日志字段，就应该设计自定义错误类型。

---

## 【代码块 3】errors.New 和 fmt.Errorf

```go
import (
    "errors"
    "fmt"
)

var ErrNotFound = errors.New("not found")

func findUser(id int) error {
    if id <= 0 {
        return fmt.Errorf("invalid user id: %d", id)
    }
    if id == 404 {
        return ErrNotFound
    }
    return nil
}

fmt.Println(findUser(-1))
fmt.Println(findUser(404))
```

---

## 【代码块 4】自定义业务错误

```go
import "fmt"

type BizError struct {
    Code int
    Msg  string
}

func (e *BizError) Error() string {
    return fmt.Sprintf("code=%d, msg=%s", e.Code, e.Msg)
}

func createOrder(amount int) error {
    if amount <= 0 {
        return &BizError{
            Code: 40001,
            Msg:  "amount must be positive",
        }
    }
    return nil
}

err := createOrder(0)
if err != nil {
    fmt.Println(err)
}
```

---

## 【文本块 7】项目里如何设计错误体系？

在真实项目里，我一般会把错误分成几类：

第一类，参数错误。
比如请求参数缺失、格式不合法、分页参数越界。

第二类，业务错误。
比如余额不足、库存不足、订单状态不允许取消。

第三类，依赖错误。
比如数据库查询失败、Redis 超时、RPC 调用失败。

第四类，系统错误。
比如配置缺失、JSON 解析失败、内部状态不一致。

工程上比较推荐统一定义一个业务错误结构：

```go
type AppError struct {
    Code    string
    Message string
    Cause   error
}
```

其中：

Code 面向程序和前端判断。
Message 面向用户或调用方。
Cause 保留底层错误，用于日志排查。

对外返回时，不要把内部堆栈、SQL、Redis key、底层网络错误直接暴露出去。
对内日志里，要保留完整上下文。

面试里可以这样回答：

我会把 error 分为业务错误和系统错误。业务错误携带稳定错误码和用户可理解信息；系统错误通常包装底层 cause，用于日志排查。Controller 或 middleware 层统一把 error 转成 HTTP 响应，避免每个 handler 到处写重复错误处理。

---

# 三、错误包装：fmt.Errorf("%w")、errors.Is、errors.As

## 【文本块 8】Q：fmt.Errorf 里的 %w 是什么？

`%w` 是 Go 1.13 引入的错误包装语法。

普通 `%v` 只是把错误格式化进字符串，原始错误关系会丢失。

而 `%w` 会把原始错误包起来，形成一条错误链。后续可以用 `errors.Is` 或 `errors.As` 判断和提取。

例如：

```go
return fmt.Errorf("query user: %w", err)
```

这表示当前错误的上层语义是 query user 失败，但底层 cause 仍然是 err。

---

## 【代码块 5】错误包装和 errors.Is

```go
import (
    "errors"
    "fmt"
)

var ErrUserNotFound = errors.New("user not found")

func queryUser(id int) error {
    return ErrUserNotFound
}

func getUserProfile(id int) error {
    if err := queryUser(id); err != nil {
        return fmt.Errorf("get user profile failed: %w", err)
    }
    return nil
}

err := getUserProfile(1)

fmt.Println(err)
fmt.Println(errors.Is(err, ErrUserNotFound))
```

---

## 【文本块 9】代码解释

虽然最终 err 的文本是：

```text
get user profile failed: user not found
```

但由于使用了 `%w`，底层仍然保留了 `ErrUserNotFound`。

所以：

```go
errors.Is(err, ErrUserNotFound)
```

可以返回 true。

如果用 `%v`，这个关系就丢了。

---

## 【文本块 10】Q：errors.Is 和 == 有什么区别？

`==` 只能判断当前 error 值本身是否等于某个 sentinel error。

`errors.Is` 会沿着错误链一层层 unwrap，判断链上有没有目标错误。

所以在现代 Go 项目里，如果错误可能被包装过，应该用 errors.Is，而不是直接 `err == ErrXXX`。

---

## 【代码块 6】== 无法识别包装后的错误

```go
import (
    "errors"
    "fmt"
)

var ErrPermissionDenied = errors.New("permission denied")

err := fmt.Errorf("check admin role: %w", ErrPermissionDenied)

fmt.Println(err == ErrPermissionDenied)
fmt.Println(errors.Is(err, ErrPermissionDenied))
```

---

## 【文本块 11】Q：errors.As 是做什么的？

`errors.As` 用于判断错误链里是否存在某种具体错误类型，如果存在，就把它提取出来。

它适合处理自定义错误类型。

例如，我们定义一个带错误码的错误类型：

```go
type BizError struct {
    Code string
    Msg  string
}
```

当错误被多层包装后，仍然可以通过 errors.As 把 BizError 提取出来。

---

## 【代码块 7】errors.As 提取自定义错误

```go
import (
    "errors"
    "fmt"
)

type BizError struct {
    Code string
    Msg  string
}

func (e *BizError) Error() string {
    return fmt.Sprintf("[%s] %s", e.Code, e.Msg)
}

func service() error {
    return &BizError{
        Code: "ORDER_NOT_FOUND",
        Msg:  "order not found",
    }
}

func handler() error {
    if err := service(); err != nil {
        return fmt.Errorf("handler failed: %w", err)
    }
    return nil
}

err := handler()

var bizErr *BizError
if errors.As(err, &bizErr) {
    fmt.Println("code:", bizErr.Code)
    fmt.Println("msg:", bizErr.Msg)
}
```

---

## 【文本块 12】errors.Is 和 errors.As 怎么选？

如果你关心的是“这个错误是不是某个特定错误值”，用 errors.Is。

例如：

```go
errors.Is(err, sql.ErrNoRows)
```

如果你关心的是“这个错误链里有没有某种错误类型，并且我要取出它的字段”，用 errors.As。

例如：

```go
var netErr net.Error
errors.As(err, &netErr)
```

面试里可以这样回答：

errors.Is 面向错误值匹配，适合 sentinel error；errors.As 面向错误类型提取，适合自定义错误类型。它们都会沿着 `%w` 包装出来的错误链向下查找。

---

# 四、sentinel error、自定义 error、opaque error

## 【文本块 13】Q：什么是 sentinel error？有什么优缺点？

sentinel error 指的是预定义的、可以被外部直接比较的错误变量。

比如标准库里的：

```go
io.EOF
sql.ErrNoRows
context.Canceled
context.DeadlineExceeded
```

我们自己也可以定义：

```go
var ErrUserNotFound = errors.New("user not found")
```

优点是简单直观，调用方可以用 errors.Is 判断。

缺点是会暴露包内部错误细节，形成 API 耦合。
一旦对外暴露了 ErrUserNotFound，其他包可能依赖它，后续修改就要考虑兼容性。

所以 sentinel error 适合稳定、明确、需要调用方分支处理的错误。

对于内部错误，更推荐包装上下文，不一定暴露具体变量。

---

## 【代码块 8】sentinel error 示例

```go
import (
    "errors"
    "fmt"
)

var ErrInsufficientBalance = errors.New("insufficient balance")

func pay(balance, amount int) error {
    if balance < amount {
        return ErrInsufficientBalance
    }
    return nil
}

err := pay(100, 200)
if errors.Is(err, ErrInsufficientBalance) {
    fmt.Println("余额不足")
}
```

---

## 【文本块 14】项目里 error 最佳实践

我通常遵循下面几条原则：

第一，错误要尽早产生，但不要过早吞掉。
底层函数发现错误就返回，上层决定是重试、降级、转换响应还是终止流程。

第二，跨层返回错误时要补充上下文。
不要直接：

```go
return err
```

而是：

```go
return fmt.Errorf("query order by id %d: %w", id, err)
```

第三，日志只在边界层打一次。
不要每一层都 log 一遍，否则同一个错误会刷很多重复日志。通常在 HTTP middleware、RPC interceptor、任务入口统一记录。

第四，对外响应和内部错误分离。
用户不应该看到数据库连接失败、SQL 语句、Redis key 等内部信息。

第五，能用 errors.Is/As 就不要字符串匹配。
字符串匹配脆弱，错误文本改了就失效。

---

# 五、panic 和 recover

## 【文本块 15】Q：panic 是什么？和 error 有什么区别？

panic 表示程序遇到了无法正常继续执行的严重问题。

它会中断当前函数的正常执行流程，开始执行当前 goroutine 栈上的 defer，然后继续向上层传播。如果一直没有被 recover，最终会导致程序崩溃。

error 是普通错误，属于预期内的失败路径。
panic 是异常控制流，通常表示不可恢复错误或程序员错误。

典型应该返回 error 的场景：

* 用户参数错误
* 查询不到数据
* 网络请求失败
* 数据库超时
* 文件不存在
* 权限不足
* 业务状态不允许

典型可以 panic 的场景：

* 程序启动时核心配置缺失
* 模板或正则表达式在 init 阶段必须编译成功
* 代码内部不变量被破坏
* 数组越界、空指针这类程序员错误
* 框架层兜底捕获 handler panic，避免整个服务崩掉

面试里最重要的是表态：

> 业务错误不要用 panic，应该返回 error。panic 更多用于不可恢复错误、程序员错误，或者框架边界兜底。

---

## 【代码块 9】panic 基本行为

```go
import "fmt"

func demo() {
    fmt.Println("before panic")
    panic("something wrong")
    fmt.Println("after panic")
}

demo()
```

---

## 【文本块 16】代码解释

`panic` 之后，当前函数的正常逻辑不会继续往下执行。

所以：

```go
fmt.Println("after panic")
```

不会执行。

在 notebook 里执行这个代码块时，当前 cell 会报 panic，这是正常现象。

---

## 【文本块 17】Q：panic 后 defer 会执行吗？

会。

panic 发生后，Go 会先执行当前 goroutine 调用栈上已经注册的 defer，再继续向上传播 panic。

这也是 recover 必须配合 defer 使用的原因。

---

## 【代码块 10】panic 后 defer 仍然执行

```go
import "fmt"

func demo() {
    defer fmt.Println("defer 1")
    defer fmt.Println("defer 2")

    fmt.Println("before panic")
    panic("boom")
}

demo()
```

---

## 【文本块 18】代码解释

defer 的执行顺序是后进先出。

所以上面会先执行 `defer 2`，再执行 `defer 1`。

然后如果没有 recover，panic 继续向上传播，最终导致程序崩溃。

---

# 六、recover

## 【文本块 19】Q：recover 是什么？为什么必须放在 defer 里？

recover 用于捕获当前 goroutine 中正在传播的 panic，让程序恢复正常执行。

但是 recover 只有在 defer 函数中直接调用时才有效。

原因是 panic 发生后，正常执行流已经中断，只有 defer 会继续执行。因此 recover 必须在 defer 中才有机会捕获 panic。

正确写法：

```go
defer func() {
    if r := recover(); r != nil {
        fmt.Println("recovered:", r)
    }
}()
```

错误写法：

```go
recover()
```

这样没有任何效果。

---

## 【代码块 11】recover 捕获 panic

```go
import "fmt"

func safeRun() {
    defer func() {
        if r := recover(); r != nil {
            fmt.Println("recovered:", r)
        }
    }()

    fmt.Println("before panic")
    panic("boom")
    fmt.Println("after panic")
}

safeRun()
fmt.Println("program continues")
```

---

## 【文本块 20】代码解释

panic 发生后，defer 函数执行。
recover 捕获 panic 后，当前函数不会继续执行 panic 后面的代码，但外层调用可以继续往下走。

所以 `after panic` 不会打印。
但 `program continues` 会打印。

面试里要说清楚：

> recover 不是让函数从 panic 那一行继续执行，而是停止 panic 继续向上传播，让外层流程恢复。

---

## 【文本块 21】Q：goroutine 里的 panic 能被主 goroutine recover 吗？

不能。

recover 只能捕获当前 goroutine 中的 panic，不能跨 goroutine 捕获。

如果子 goroutine panic，而它自己没有 recover，整个程序仍然可能崩溃。

所以在生产项目中，只要你启动了 goroutine，尤其是后台任务、异步 worker、消息消费者，就应该在 goroutine 入口处加 recover 兜底。

---

## 【代码块 12】主 goroutine 不能 recover 子 goroutine 的 panic

```go
import (
    "fmt"
    "time"
)

func mainLike() {
    defer func() {
        if r := recover(); r != nil {
            fmt.Println("main recovered:", r)
        }
    }()

    go func() {
        panic("panic in child goroutine")
    }()

    time.Sleep(time.Second)
}

mainLike()
```

---

## 【文本块 22】代码解释】

这个例子中，外层 defer 不能捕获子 goroutine 的 panic。

在 notebook 里直接执行可能导致 cell 报错。工程里一定要记住：每个 goroutine 都应该自己兜底 recover。

---

## 【代码块 13】goroutine 内部 recover

```go
import (
    "fmt"
    "time"
)

func goSafe(fn func()) {
    go func() {
        defer func() {
            if r := recover(); r != nil {
                fmt.Println("goroutine recovered:", r)
            }
        }()
        fn()
    }()
}

goSafe(func() {
    panic("worker panic")
})

time.Sleep(100 * time.Millisecond)
fmt.Println("program continues")
```

---

## 【文本块 23】项目里 panic/recover 怎么用？

在 Web 服务里，通常会在 middleware 层统一 recover。

比如：

* Gin Recovery middleware
* gRPC unary interceptor
* HTTP middleware
* 消息消费者 worker 入口
* 定时任务入口

这样即使某个请求 handler panic，也不会导致整个进程退出。

但要注意，recover 之后不能假装什么都没发生。应该至少做三件事：

第一，记录日志。
包括 panic 值、堆栈、请求信息、trace_id、用户 ID 等。

第二，返回稳定错误。
HTTP 场景通常返回 500。RPC 场景返回 internal error。

第三，必要时告警。
panic 通常表示程序员错误或未预期状态，不能静默吞掉。

面试里可以这样回答：

业务代码里不应该依赖 panic 做流程控制，但在框架边界要用 recover 做兜底保护。比如 HTTP middleware 捕获 handler panic，记录堆栈并返回 500，避免单个请求打崩整个服务。

---

# 七、defer

## 【文本块 24】Q：defer 的作用是什么？

defer 用于延迟执行函数，通常用来释放资源、解锁、关闭文件、recover panic。

常见场景：

```go
file, err := os.Open(path)
if err != nil {
    return err
}
defer file.Close()
```

或者：

```go
mu.Lock()
defer mu.Unlock()
```

defer 会在当前函数返回前执行。
如果有多个 defer，按照后进先出顺序执行。
defer 的参数会在注册 defer 时求值。

---

## 【代码块 14】defer 后进先出

```go
import "fmt"

func demoDefer() {
    defer fmt.Println("first")
    defer fmt.Println("second")
    defer fmt.Println("third")
}

demoDefer()
```

---

## 【代码块 15】defer 参数会提前求值

```go
import "fmt"

func demo() {
    x := 1
    defer fmt.Println("defer x:", x)

    x = 2
    fmt.Println("current x:", x)
}

demo()
```

---

## 【文本块 25】代码解释

defer 注册时，`fmt.Println("defer x:", x)` 的参数 x 已经被求值为 1。

所以最后 defer 打印的是 1，不是 2。

如果想让 defer 看到最新值，可以用闭包：

```go
defer func() {
    fmt.Println(x)
}()
```

闭包里读取的是变量本身，执行时会拿到最新值。

---

## 【代码块 16】defer 闭包读取最新变量

```go
import "fmt"

func demo() {
    x := 1

    defer func() {
        fmt.Println("defer x:", x)
    }()

    x = 2
    fmt.Println("current x:", x)
}

demo()
```

---

## 【文本块 26】defer 工程注意事项

第一，在循环里 defer 要小心。
如果循环很多次，每次 defer 都要等函数结束才执行，可能导致资源迟迟不释放。

错误示例：

```go
for _, path := range paths {
    f, _ := os.Open(path)
    defer f.Close()
}
```

如果文件很多，就可能同时打开大量文件。

可以把循环体封装成函数，让 defer 尽快执行。

第二，defer unlock 很常见，但要确保 lock 成功后再 defer。
不要在 lock 之前的代码可能 panic 或 return 的情况下乱写 unlock。

第三，defer 有少量开销。
现代 Go 版本已经优化了 defer 性能，普通业务代码优先可读性，不要过度手动优化。只有极端热点路径才考虑不用 defer。

---

# 八、reflect 反射基础

## 【文本块 27】Q：Go 反射是什么？有什么用？

反射是程序在运行时检查和操作类型信息和值信息的能力。

Go 反射主要由 reflect 包提供，核心是两个类型：

```go
reflect.Type
reflect.Value
```

Type 表示类型信息。
Value 表示运行时的值。

反射常见用途：

* JSON 序列化和反序列化
* ORM 字段映射
* 参数校验
* 配置文件绑定
* 依赖注入
* RPC 框架参数编解码
* 通用工具函数

比如 encoding/json 能根据结构体字段和 tag 自动生成 JSON，底层就大量使用了反射。

面试里可以这样回答：

Go 反射是运行时获取类型和值信息的机制，核心是 reflect.Type 和 reflect.Value。它让框架可以在不知道具体类型的情况下，动态读取结构体字段、方法、tag，并进行赋值或调用。但反射有性能开销，也会牺牲类型安全，所以业务代码里不应该滥用。

---

## 【代码块 17】reflect.TypeOf 和 reflect.ValueOf

```go
import (
    "fmt"
    "reflect"
)

x := 123

t := reflect.TypeOf(x)
v := reflect.ValueOf(x)

fmt.Println("type:", t)
fmt.Println("kind:", t.Kind())
fmt.Println("value:", v)
fmt.Println("value kind:", v.Kind())
```

---

## 【文本块 28】Type 和 Kind 有什么区别？

Type 表示具体类型。

Kind 表示底层分类。

例如：

```go
type MyInt int
```

对于 MyInt：

Type 是 `main.MyInt`。
Kind 是 `int`。

所以 Type 更具体，Kind 更抽象。

面试里可以这样说：

> Type 是类型本身，带有包路径和类型名；Kind 是类型的底层类别，比如 int、struct、slice、map、ptr。多个不同的自定义类型可能有相同的 Kind。

---

## 【代码块 18】Type 和 Kind 的区别

```go
import (
    "fmt"
    "reflect"
)

type UserID int64

var id UserID = 100

t := reflect.TypeOf(id)

fmt.Println("type:", t)
fmt.Println("kind:", t.Kind())
```

---

# 九、reflect.Value 修改值

## 【文本块 29】Q：为什么 reflect 修改值时必须传指针？

因为 Go 是值传递。

如果你传入普通变量：

```go
reflect.ValueOf(x)
```

反射拿到的是 x 的副本，不是原变量本身。这个 Value 不可设置。

如果想修改原变量，必须传指针：

```go
reflect.ValueOf(&x).Elem()
```

其中：

* `&x` 让反射拿到变量地址
* `Elem()` 取指针指向的元素
* 这个元素才是可设置的

可以通过 `CanSet()` 判断一个 Value 是否可修改。

---

## 【代码块 19】不可设置的 reflect.Value

```go
import (
    "fmt"
    "reflect"
)

x := 10

v := reflect.ValueOf(x)

fmt.Println("CanSet:", v.CanSet())

// v.SetInt(20) // 这里会 panic
```

---

## 【代码块 20】通过指针修改值

```go
import (
    "fmt"
    "reflect"
)

x := 10

v := reflect.ValueOf(&x).Elem()

fmt.Println("CanSet:", v.CanSet())

v.SetInt(20)

fmt.Println(x)
```

---

## 【文本块 30】追问：Elem 是什么？

Elem 可以理解成“解引用”。

如果 Value 是指针，Elem 返回指针指向的值。
如果 Value 是 interface，Elem 返回 interface 中装的动态值。

常见写法：

```go
v := reflect.ValueOf(&x).Elem()
```

这表示从 `*int` 解引用到 int 本身。

---

# 十、反射读取 struct 字段和 tag

## 【文本块 31】Q：Go 结构体 tag 是怎么被读取的？

结构体 tag 本质上是一段写在字段后面的元信息字符串。

例如：

```go
type User struct {
    Name string `json:"name" db:"user_name"`
}
```

Go 编译器会把 tag 保存到类型元信息中。运行时可以通过反射读取。

encoding/json、GORM、validator 等框架就是通过反射读取 tag，然后决定字段怎么序列化、映射或校验。

---

## 【代码块 21】读取 struct 字段和 tag

```go
import (
    "fmt"
    "reflect"
)

type User struct {
    ID   int    `json:"id" db:"id"`
    Name string `json:"name" db:"user_name"`
}

t := reflect.TypeOf(User{})

for i := 0; i < t.NumField(); i++ {
    field := t.Field(i)

    fmt.Println("field name:", field.Name)
    fmt.Println("field type:", field.Type)
    fmt.Println("json tag:", field.Tag.Get("json"))
    fmt.Println("db tag:", field.Tag.Get("db"))
    fmt.Println("---")
}
```

---

## 【文本块 32】追问：为什么反射不能直接修改未导出字段？

Go 的反射遵循语言的可见性规则。

结构体字段如果是小写开头，属于未导出字段。即使用反射拿到了它，也不能随便修改。

这是为了保持封装性和安全性。

当然，unsafe 可以绕过限制，但业务代码里不推荐这么做。

面试里可以这样回答：

反射虽然强大，但不是完全无视 Go 语言规则。对于未导出字段，reflect 默认不能 Set，这是 Go 为了保证包封装边界。

---

## 【代码块 22】反射修改 struct 字段

```go
import (
    "fmt"
    "reflect"
)

type User struct {
    Name string
    Age  int
}

u := User{Name: "Tom", Age: 18}

v := reflect.ValueOf(&u).Elem()

nameField := v.FieldByName("Name")
if nameField.IsValid() && nameField.CanSet() {
    nameField.SetString("Jerry")
}

ageField := v.FieldByName("Age")
if ageField.IsValid() && ageField.CanSet() {
    ageField.SetInt(20)
}

fmt.Println(u)
```

---

# 十一、反射调用方法

## 【文本块 33】Q：Go 反射可以调用方法吗？

可以。

可以通过 `MethodByName` 找到方法，再通过 `Call` 调用。

不过反射调用方法性能较差，也没有编译期类型检查，业务代码里不推荐频繁使用。它更多用于框架、插件、测试工具、RPC 分发等场景。

---

## 【代码块 23】反射调用方法

```go
import (
    "fmt"
    "reflect"
)

type Calculator struct{}

func (Calculator) Add(a, b int) int {
    return a + b
}

c := Calculator{}

v := reflect.ValueOf(c)
method := v.MethodByName("Add")

args := []reflect.Value{
    reflect.ValueOf(10),
    reflect.ValueOf(20),
}

results := method.Call(args)

fmt.Println(results[0].Int())
```

---

## 【文本块 34】反射的缺点

反射的缺点主要有三个：

第一，性能较差。
反射需要运行时解析类型和值，不能像普通代码那样充分利用编译期优化。

第二，类型安全弱。
很多错误要到运行时才暴露，比如字段名写错、类型不匹配、参数数量错误。

第三，可读性和可维护性差。
反射代码通常比较绕，新人不容易理解。

所以项目里一般遵循：

* 能用普通接口就不用反射
* 能用泛型就不用反射
* 框架底层可以用反射
* 业务核心逻辑谨慎使用反射

面试里可以这样总结：

反射是 Go 提供给框架和通用工具的能力，不是日常业务代码的首选。它适合做类型未知的动态处理，但代价是性能、类型安全和可读性下降。

---

# 十二、泛型基础

## 【文本块 35】Q：Go 泛型是什么？解决了什么问题？

Go 1.18 引入了泛型。

泛型的核心作用是：让函数、结构体、接口可以使用类型参数，从而在保持类型安全的前提下复用代码。

没有泛型之前，如果我们想写一个通用函数，要么为每种类型写一份，要么用 interface{}，要么用反射。

比如求两个 int 的最大值：

```go
func MaxInt(a, b int) int
```

求两个 float64 的最大值：

```go
func MaxFloat64(a, b float64) float64
```

逻辑一样，但类型不同，就要重复写。

有泛型之后，可以写成：

```go
func Max[T Ordered](a, b T) T
```

这样同一份代码可以用于 int、float64、string 等支持比较的类型。

面试里可以这样回答：

Go 泛型通过类型参数和类型约束，让我们能写出类型安全的通用代码。它主要解决容器、算法、工具函数中大量重复代码的问题，比 interface{} 更安全，比反射性能和可读性更好。

---

## 【代码块 24】最简单的泛型函数

```go
import "fmt"

func First[T any](items []T) T {
    return items[0]
}

fmt.Println(First([]int{1, 2, 3}))
fmt.Println(First([]string{"a", "b", "c"}))
```

---

## 【文本块 36】any 是什么？

any 是 interface{} 的别名。

也就是说：

```go
type any = interface{}
```

它表示任意类型。

在泛型里，`T any` 表示类型参数 T 可以是任意类型。

不过 any 只是“没有约束”，不代表你可以对 T 做任何操作。

例如：

```go
func Add[T any](a, b T) T {
    return a + b
}
```

这不能编译，因为 any 没有保证 T 支持 `+` 运算。

---

# 十三、类型约束

## 【文本块 37】Q：什么是类型约束？

类型约束用于限制泛型类型参数必须满足什么能力。

比如，如果一个泛型函数里要使用 `==`，那么 T 必须是可比较类型：

```go
func Contains[T comparable](items []T, target T) bool
```

如果要使用 `<`，就必须约束 T 是有序类型，比如 int、float、string。

Go 标准库提供了 comparable 这个预声明约束，但没有内置 Ordered。通常可以自己定义。

---

## 【代码块 25】comparable 约束

```go
import "fmt"

func Contains[T comparable](items []T, target T) bool {
    for _, item := range items {
        if item == target {
            return true
        }
    }
    return false
}

fmt.Println(Contains([]int{1, 2, 3}, 2))
fmt.Println(Contains([]string{"go", "java"}, "python"))
```

---

## 【文本块 38】comparable 是什么？

comparable 表示这个类型的值可以使用 `==` 和 `!=` 比较。

它常用于：

* map key
* 去重
* contains
* set
* equal 判断

注意，slice、map、function 不满足 comparable。

所以：

```go
Contains([][]int{{1}, {2}}, []int{1})
```

不能编译。

这是好事，因为编译期就阻止了不合法比较。

---

## 【代码块 26】自定义 Ordered 约束

```go
import "fmt"

type Ordered interface {
    ~int | ~int64 | ~float64 | ~string
}

func Max[T Ordered](a, b T) T {
    if a > b {
        return a
    }
    return b
}

fmt.Println(Max(1, 2))
fmt.Println(Max(3.14, 2.71))
fmt.Println(Max("go", "java"))
```

---

## 【文本块 39】~T 是什么意思？

`~T` 表示底层类型是 T 的所有类型。

例如：

```go
~int
```

不仅包括 int，也包括底层类型是 int 的自定义类型：

```go
type MyInt int
```

如果约束写成：

```go
interface {
    int
}
```

那只允许 int，不允许 MyInt。

如果写成：

```go
interface {
    ~int
}
```

那 int 和 MyInt 都允许。

---

## 【代码块 27】~ 支持底层类型匹配

```go
import "fmt"

type MyInt int

type IntLike interface {
    ~int
}

func Double[T IntLike](x T) T {
    return x * 2
}

var a int = 10
var b MyInt = 20

fmt.Println(Double(a))
fmt.Println(Double(b))
```

---

# 十四、泛型结构体

## 【文本块 40】Q：Go 支持泛型结构体吗？

支持。

常见例子是实现一个栈、队列、集合、可选值、响应包装类型等。

比如：

```go
type Stack[T any] struct {
    items []T
}
```

这样可以创建：

```go
Stack[int]
Stack[string]
Stack[*User]
```

相比 interface{}，泛型结构体有编译期类型检查，不需要调用方做类型断言。

---

## 【代码块 28】泛型 Stack

```go
import "fmt"

type Stack[T any] struct {
    items []T
}

func (s *Stack[T]) Push(v T) {
    s.items = append(s.items, v)
}

func (s *Stack[T]) Pop() (T, bool) {
    var zero T

    if len(s.items) == 0 {
        return zero, false
    }

    last := len(s.items) - 1
    v := s.items[last]
    s.items = s.items[:last]

    return v, true
}

st := Stack[int]{}
st.Push(1)
st.Push(2)

v, ok := st.Pop()
fmt.Println(v, ok)
```

---

## 【文本块 41】泛型里的零值怎么处理？

在泛型函数或泛型结构体里，如果你要返回 T 的零值，常见写法是：

```go
var zero T
return zero
```

因为你不知道 T 具体是什么类型，不能直接写 0、""、nil。

例如：

```go
func Pop() (T, bool) {
    var zero T
    if empty {
        return zero, false
    }
}
```

这个写法在泛型容器里非常常见。

---

# 十五、泛型、interface、reflect 怎么选？

## 【文本块 42】Q：泛型、interface、reflect 分别适合什么场景？

这是 Go 1.18 之后非常重要的工程判断题。

第一，interface 适合表达行为抽象。
比如：

```go
type Reader interface {
    Read(p []byte) (n int, err error)
}
```

你不关心对方是什么类型，只关心它有没有 Read 行为。

第二，泛型适合表达类型参数化。
比如通用的 List、Set、Stack、Map、Max、Min、Contains。你关心的是“同一套逻辑适配多种类型”，并且希望保持类型安全。

第三，reflect 适合运行时动态处理未知类型。
比如 JSON、ORM、配置解析、参数校验。编译期不知道具体结构，只能运行时读取字段、tag、类型。

面试里可以这样回答：

如果抽象的是行为，用 interface；如果抽象的是数据类型但逻辑相同，用泛型；如果类型在编译期未知，需要运行时读取字段和 tag，用 reflect。业务代码优先 interface 和泛型，反射更多留给框架底层。

---

## 【文本块 43】泛型会不会替代 interface？

不会。

泛型和 interface 解决的问题不同。

interface 是动态多态，强调“这个对象能做什么”。
泛型是静态类型参数，强调“这段逻辑可以适用于哪些类型”。

例如 io.Reader 就非常适合 interface：

```go
func ReadAll(r io.Reader)
```

如果改成泛型反而没有意义。

而像 Contains、Max、Stack 这种算法和容器，更适合泛型。

---

# 十六、Go IO 抽象

## 【文本块 44】Q：Go 的 IO 模型核心是什么？

Go 标准库 IO 的核心是两个接口：

```go
type Reader interface {
    Read(p []byte) (n int, err error)
}

type Writer interface {
    Write(p []byte) (n int, err error)
}
```

这两个接口非常简单，但非常强大。

只要一个类型实现了 Read 方法，它就可以作为数据源。
只要一个类型实现了 Write 方法，它就可以作为数据目的地。

例如：

* os.File 实现了 Reader 和 Writer
* bytes.Buffer 实现了 Reader 和 Writer
* strings.Reader 实现了 Reader
* net.Conn 实现了 Reader 和 Writer
* HTTP response body 是 Reader
* HTTP request body 是 Reader

这让 Go 的 IO 代码非常统一。

比如：

```go
io.Copy(dst, src)
```

只要求 dst 是 Writer，src 是 Reader，不关心它们具体是文件、网络连接、内存 buffer 还是压缩流。

面试里可以这样回答：

Go IO 的核心是面向接口的组合。Reader 表示可读数据源，Writer 表示可写数据目标。标准库大量类型都实现了这两个接口，所以文件、网络、内存、压缩、加密等 IO 操作可以通过统一的 io.Reader/io.Writer 管道组合起来。

---

## 【代码块 29】io.Reader 基本使用

```go
import (
    "fmt"
    "io"
    "strings"
)

r := strings.NewReader("hello go")

buf := make([]byte, 4)

for {
    n, err := r.Read(buf)
    if n > 0 {
        fmt.Println("read:", string(buf[:n]))
    }
    if err == io.EOF {
        break
    }
    if err != nil {
        fmt.Println("error:", err)
        break
    }
}
```

---

## 【文本块 45】为什么 Read 返回 n 和 err？

Read 的签名是：

```go
Read(p []byte) (n int, err error)
```

它返回两个信息：

n 表示本次读取了多少字节。
err 表示读取过程中遇到的错误。

注意，n > 0 和 err != nil 可能同时出现。

所以正确处理方式通常是：

```go
if n > 0 {
    // 先处理数据
}
if err == io.EOF {
    break
}
if err != nil {
    return err
}
```

不能一看到 err != nil 就立刻丢掉 n 字节数据。

---

## 【文本块 46】Q：io.EOF 是什么？

io.EOF 表示读到数据结尾。

它不是严重错误，而是一个结束信号。

例如从文件或 strings.Reader 读取时，如果没有更多数据了，就会返回 io.EOF。

工程里通常这样处理：

```go
if err == io.EOF {
    break
}
```

不要把 EOF 当成系统异常直接打 error 日志。

---

# 十七、io.Copy

## 【文本块 47】Q：io.Copy 有什么用？

io.Copy 用于把一个 Reader 的数据复制到一个 Writer。

签名是：

```go
func Copy(dst Writer, src Reader) (written int64, err error)
```

它体现了 Go IO 抽象的威力。

你可以从文件复制到文件。
可以从 HTTP response body 复制到文件。
可以从字符串复制到 buffer。
可以从网络连接复制到另一个网络连接。

只要源实现 Reader，目标实现 Writer，就可以组合。

---

## 【代码块 30】io.Copy 示例

```go
import (
    "bytes"
    "fmt"
    "io"
    "strings"
)

src := strings.NewReader("hello io.Copy")
var dst bytes.Buffer

n, err := io.Copy(&dst, src)
if err != nil {
    fmt.Println("copy error:", err)
} else {
    fmt.Println("written:", n)
    fmt.Println("content:", dst.String())
}
```

---

## 【文本块 48】bytes.Buffer 和 strings.Builder 的区别

bytes.Buffer 主要处理字节数据，既可以读，也可以写，实现了 Reader 和 Writer。

strings.Builder 主要用于高效构建字符串，只写不读，更适合字符串拼接。

如果最终目标是 string，并且只是不断拼接文本，用 strings.Builder。

如果你处理的是 []byte，或者需要作为 Reader/Writer 参与 IO 管道，用 bytes.Buffer。

---

# 十八、bufio

## 【文本块 49】Q：bufio 是干什么的？为什么能提高性能？

bufio 提供带缓冲的 IO。

很多底层 IO 操作，比如磁盘、网络，系统调用成本比较高。如果每读一个字节、写一个字节都直接调用底层 IO，会非常低效。

bufio.Reader 会一次从底层读取较大块数据放到内存缓冲区，应用每次小读时先从缓冲区拿。

bufio.Writer 会先把小写入积累到缓冲区，等缓冲区满了或者手动 Flush 时，再一次性写到底层。

所以 bufio 的核心价值是减少系统调用次数，提高小块读写性能。

---

## 【代码块 31】bufio.Reader 按行读取

```go
import (
    "bufio"
    "fmt"
    "io"
    "strings"
)

data := "line1\nline2\nline3\n"
r := bufio.NewReader(strings.NewReader(data))

for {
    line, err := r.ReadString('\n')
    if len(line) > 0 {
        fmt.Print("line: ", line)
    }
    if err == io.EOF {
        break
    }
    if err != nil {
        fmt.Println("error:", err)
        break
    }
}
```

---

## 【代码块 32】bufio.Writer 需要 Flush

```go
import (
    "bufio"
    "bytes"
    "fmt"
)

var buf bytes.Buffer

w := bufio.NewWriter(&buf)

w.WriteString("hello")
w.WriteString(" ")
w.WriteString("bufio")

fmt.Println("before flush:", buf.String())

w.Flush()

fmt.Println("after flush:", buf.String())
```

---

## 【文本块 50】追问：bufio.Writer 为什么必须 Flush？

因为 WriteString 只是写到内存缓冲区，不一定立刻写到底层 Writer。

如果程序结束前没有 Flush，缓冲区里的数据可能还没真正写出去。

所以使用 bufio.Writer 时，常见写法是：

```go
w := bufio.NewWriter(file)
defer w.Flush()
```

不过如果 Flush 可能返回错误，严格来说应该显式处理：

```go
if err := w.Flush(); err != nil {
    return err
}
```

---

# 十九、文件 IO

## 【文本块 51】Q：Go 里怎么读写文件？

Go 文件操作主要在 os 包里。

常见函数：

```go
os.Open
os.Create
os.OpenFile
os.ReadFile
os.WriteFile
```

如果是小文件，可以直接用 `os.ReadFile` 一次性读取。

如果是大文件，应该用流式读取，例如 bufio.Reader 或者分块 Read，避免一次性把整个文件加载到内存。

写文件时，也要注意权限、覆盖、追加、Flush、Close。

---

## 【代码块 33】读取小文件

```go
import (
    "fmt"
    "os"
)

err := os.WriteFile("demo.txt", []byte("hello file"), 0644)
if err != nil {
    fmt.Println("write error:", err)
}

data, err := os.ReadFile("demo.txt")
if err != nil {
    fmt.Println("read error:", err)
} else {
    fmt.Println(string(data))
}
```

---

## 【代码块 34】流式读取文件

```go
import (
    "bufio"
    "fmt"
    "io"
    "os"
)

f, err := os.Open("demo.txt")
if err != nil {
    fmt.Println("open error:", err)
} else {
    defer f.Close()

    r := bufio.NewReader(f)
    for {
        line, err := r.ReadString('\n')
        if len(line) > 0 {
            fmt.Print(line)
        }
        if err == io.EOF {
            break
        }
        if err != nil {
            fmt.Println("read error:", err)
            break
        }
    }
}
```

---

## 【文本块 52】defer Close 要注意什么？

文件打开后要及时关闭，否则可能导致文件描述符泄漏。

常见写法：

```go
f, err := os.Open(path)
if err != nil {
    return err
}
defer f.Close()
```

但是严格工程里，Close 也可能返回错误。对于只读文件，Close 错误通常不关键；对于写文件，Close 或 Flush 失败可能意味着数据没真正落盘，应该处理。

写文件时更严谨：

```go
if err := w.Flush(); err != nil {
    return err
}
if err := f.Close(); err != nil {
    return err
}
```

---

# 二十、网络 IO 与 net.Conn

## 【文本块 53】Q：net.Conn 是什么？

net.Conn 是 Go 网络编程里的核心接口之一。

它代表一个通用的网络连接，通常 TCP 连接会实现它。

net.Conn 同时实现了 Reader 和 Writer：

```go
Read(b []byte) (n int, err error)
Write(b []byte) (n int, err error)
Close() error
SetDeadline(t time.Time) error
SetReadDeadline(t time.Time) error
SetWriteDeadline(t time.Time) error
```

这意味着网络连接可以直接参与 io.Copy、bufio.Reader、bufio.Writer 等标准 IO 组合。

---

## 【文本块 54】Q：Go 网络 IO 是阻塞还是非阻塞？

从用户代码视角看，Go 网络 IO 是阻塞写法。

例如：

```go
n, err := conn.Read(buf)
```

如果没有数据，当前 goroutine 会等待。

但从 runtime 底层看，Go 并不会让操作系统线程傻等在网络 IO 上。

Go runtime 使用 netpoller，把网络 fd 设置为非阻塞，并基于 epoll/kqueue/IOCP 等机制监听事件。

当某个 goroutine 读网络时，如果数据没准备好，这个 goroutine 会被挂起，底层线程可以去执行别的 goroutine。等网络事件就绪后，runtime 再把这个 goroutine 放回可运行队列。

所以 Go 提供了同步阻塞的编程模型，但底层通过 runtime netpoller 实现高并发。

面试里可以这样回答：

Go 网络 IO 在代码层面是阻塞式调用，但 runtime 底层使用非阻塞 fd 和 netpoller。当 goroutine 等待网络事件时，不会长期占用 OS 线程，runtime 会调度其他 goroutine 执行。因此 Go 可以用 goroutine-per-connection 的简单模型支撑大量连接。

---

# 二十一、context 与 IO 超时

## 【文本块 55】Q：Go IO 中如何控制超时？

常见方式有两类。

第一，net.Conn 层面设置 deadline：

```go
conn.SetReadDeadline(time.Now().Add(3 * time.Second))
```

这适合底层连接读写超时控制。

第二，使用 context。

在 HTTP、数据库、RPC 等高层 API 里，通常都支持 context：

```go
req, _ := http.NewRequestWithContext(ctx, method, url, body)
db.QueryContext(ctx, query)
grpcCall(ctx, req)
```

context 可以统一传递超时、取消信号和请求级元信息。

工程里一般推荐：外层请求入口创建 context，向下传递到 DB、Redis、RPC、HTTP client 等所有依赖调用。

---

## 【代码块 35】context timeout 示例

```go
import (
    "context"
    "fmt"
    "time"
)

ctx, cancel := context.WithTimeout(context.Background(), 100*time.Millisecond)
defer cancel()

select {
case <-time.After(500 * time.Millisecond):
    fmt.Println("done")
case <-ctx.Done():
    fmt.Println("timeout:", ctx.Err())
}
```

---

## 【文本块 56】context.Canceled 和 context.DeadlineExceeded 区别

`context.Canceled` 表示 context 被主动取消。

例如调用了：

```go
cancel()
```

`context.DeadlineExceeded` 表示超过了 deadline 或 timeout。

例如：

```go
context.WithTimeout(...)
```

时间到了自动取消。

这两个都是 sentinel error，可以用 errors.Is 判断：

```go
errors.Is(err, context.Canceled)
errors.Is(err, context.DeadlineExceeded)
```

---

# 二十二、Go IO 和 Java BIO/NIO 的对比

## 【文本块 57】Q：Go 的网络模型和 Java NIO/Netty 有什么区别？

Java BIO 常见模型是一个连接一个线程，线程阻塞读写。连接多了之后，线程数会膨胀，上下文切换和内存开销变大。

Java NIO/Netty 使用 Selector/Reactor 模型，让少量线程管理大量连接。业务代码通常要理解事件循环、Channel、Buffer、Pipeline 等概念。

Go 的思路不同。

Go 让开发者用同步阻塞的方式写代码：

```go
for {
    n, err := conn.Read(buf)
}
```

但 runtime 底层通过 goroutine 调度器和 netpoller 把网络等待异步化。

所以 Go 的优势是：

* 编程模型接近同步代码，简单直观
* goroutine 比 OS thread 轻量
* 网络等待不会长期占用线程
* 不需要业务层直接写复杂 Reactor 代码
* 和 context、channel、select 组合方便

但 Go 也不是没有成本。

如果每个连接、每个请求都启动 goroutine，也要注意 goroutine 泄漏、内存占用、连接生命周期、超时取消和背压控制。

面试里可以这样回答：

Java NIO/Netty 是业务层显式使用 Reactor 模型；Go 是语言 runtime 层帮我们封装了 netpoller 和 goroutine 调度。开发者可以用同步阻塞风格写网络代码，但底层不会简单地一个阻塞 IO 占死一个线程。这是 Go 网络编程简单且高并发的重要原因。

---

# 二十三、本章高频面试题速记

## 【文本块 58】error 速记

Go 使用显式 error 返回，不使用 Java 式 try-catch 作为常规错误处理。

error 是接口：

```go
type error interface {
    Error() string
}
```

error 为 nil 的前提是接口动态类型和动态值都为 nil。

创建 error 常见方式：

* errors.New
* fmt.Errorf
* 自定义 error 类型

`fmt.Errorf("%w", err)` 可以包装错误链。
`errors.Is` 用于判断错误链中是否有某个错误值。
`errors.As` 用于提取错误链中的某种错误类型。

项目里不要字符串匹配错误。
不要每一层都打日志。
不要把内部错误直接暴露给用户。
业务错误要有稳定错误码。
系统错误要保留 cause 和上下文。

---

## 【文本块 59】panic/recover 速记

error 是预期内错误。
panic 是不可恢复错误或程序员错误。
业务流程不要用 panic 控制。
panic 后 defer 会执行。
recover 只有在 defer 中直接调用才有效。
recover 只能捕获当前 goroutine 的 panic，不能跨 goroutine。
启动 goroutine 时，最好在入口处做 recover 兜底。
Web/RPC/消息消费者应在边界层统一 recover、打日志、返回稳定错误。

---

## 【文本块 60】reflect 速记

反射核心是：

```go
reflect.Type
reflect.Value
```

Type 是具体类型。
Kind 是底层类别。
Value 表示运行时值。
修改值必须传指针并 Elem。
CanSet 判断是否可修改。
结构体 tag 可以通过反射读取。
JSON、ORM、validator、配置绑定大量依赖反射。
反射性能较差、类型安全弱、可读性差，业务代码不要滥用。

---

## 【文本块 61】泛型速记

Go 1.18 引入泛型。

泛型核心是类型参数和类型约束。

`any` 等于 `interface{}`。
`comparable` 表示可使用 == 和 !=。
`~T` 表示底层类型是 T 的类型集合。
泛型适合容器、算法、工具函数。
interface 适合行为抽象。
reflect 适合运行时动态处理未知类型。
泛型不会替代 interface，二者解决的问题不同。

---

## 【文本块 62】IO 速记

Go IO 核心是：

```go
io.Reader
io.Writer
```

Reader 表示数据源。
Writer 表示数据目的地。
io.Copy 把 Reader 复制到 Writer。
bufio 通过缓冲减少系统调用。
bytes.Buffer 适合字节数据和 IO 管道。
strings.Builder 适合高效构建字符串。
os.File、net.Conn、bytes.Buffer 都能和 Reader/Writer 体系组合。
io.EOF 表示读到结尾，不是严重错误。
Go 网络 IO 代码层面是阻塞风格，runtime 底层用 netpoller 支撑高并发。
context 用于超时、取消和请求级信息传递。

---

# 二十四、本章最终面试回答模板

## 【文本块 63】综合回答模板

如果面试官让我整体讲 Go 的 error、panic、reflect、泛型和 IO，我会这样回答：

Go 的错误处理核心是显式返回 error。error 本质是接口，只要实现 Error() string 就可以作为错误返回。相比 Java 的异常机制，Go 更强调错误路径在代码中显式可见。项目里我通常会用自定义错误类型承载错误码和用户提示，用 fmt.Errorf("%w") 包装底层错误，并通过 errors.Is、errors.As 在上层判断和提取。日志一般只在服务边界统一打印，避免重复记录。

panic 不是常规业务错误处理方式，它更适合不可恢复错误或程序员错误。panic 发生后会执行 defer，recover 必须在 defer 中调用才有效，而且只能捕获当前 goroutine 的 panic。工程里通常会在 HTTP middleware、gRPC interceptor、消息消费者和后台 goroutine 入口做 recover 兜底，记录堆栈并返回稳定错误。

reflect 是 Go 的运行时类型和值操作机制，核心是 reflect.Type 和 reflect.Value。Type 表示具体类型，Kind 表示底层分类。反射可以读取结构体字段、tag，也可以动态赋值和调用方法，所以 JSON、ORM、配置解析、参数校验等框架大量使用反射。但反射性能较差、类型安全弱，所以业务代码里要谨慎使用。

泛型是 Go 1.18 引入的类型参数机制，主要解决容器、算法、工具函数的重复代码问题。any 表示任意类型，comparable 表示可比较类型，~T 表示底层类型匹配。泛型适合类型参数化，interface 适合行为抽象，reflect 适合运行时未知类型处理，三者不是互相替代关系。

Go IO 的核心是 io.Reader 和 io.Writer。标准库通过这两个小接口把文件、网络、内存 buffer、字符串、压缩流等统一起来。io.Copy 可以把任意 Reader 的数据复制到任意 Writer。bufio 通过缓冲减少系统调用。Go 网络 IO 在代码层面是阻塞风格，但 runtime 底层通过非阻塞 fd 和 netpoller 挂起等待 IO 的 goroutine，从而用简单的 goroutine-per-connection 模型支撑高并发。

一句话总结：Go 的基础设计强调显式、组合和简单抽象。error 让失败路径显式，defer 保证资源释放，recover 做边界兜底，reflect 支撑框架动态能力，泛型补齐类型安全的代码复用，Reader/Writer 统一了整个 IO 生态。
